# Introduction
This notebook uses custom build classifiers to source new species predictions for images where confAnimal > 0.75


In [1]:
# Data Handling
import pandas as pd

# IO - getting files and images
from pymongo import MongoClient
from kaggle_secrets import UserSecretsClient
import requests
import json
import os
from urllib.parse import urlparse

from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from PIL import Image
from io import BytesIO

# For randomizing which images get downloaded
import random
from tqdm.auto import tqdm

# Specific to fastai and model selection
import timm
from fastai.vision.all import *

print("==== Loaded Libraries ====")

==== Loaded Libraries ====


# Accessing observations (image labels) through pyMongo
This version uses pymongo (MongoClient) 

In [2]:
%%time
# Get the stored mongo uri secret
user_secrets = UserSecretsClient()
mongo_uri = user_secrets.get_secret("MONGO_PROD")

# Access the server
client = MongoClient(mongo_uri)
db = client['test']
collection = db['cameratrapmedias']

CPU times: user 513 ms, sys: 115 ms, total: 628 ms
Wall time: 1.03 s


We're going to allow an aiResults filter at this stage to include only images probably with something in them - this will drop our ability to detect blanks though

In [28]:
%%time
# Grab results where aiResults > 0.75 vs total
MIN_CONF_ANIMAL = 0.75
MAX_CONF_ANIMAL = 1.00

def fetch_all_obs():
    query = {"aiResults.0.confAnimal": {"$gte": MIN_CONF_ANIMAL, "$lte": MAX_CONF_ANIMAL}}
    projection = {
        "_id": 0,
        "mediaID": 1,
        "publicURL": 1,
    }
    all_obs = list(collection.find(query, projection))
    print(f"Retrieved {len(all_obs)} documents with aiResults.confAnimal > {MIN_CONF_ANIMAL} and < {MAX_CONF_ANIMAL}.")
    return all_obs

# Try the fetch operation
try:
    print("===== Starting MongoDB Fetch =====")
    obs_json = fetch_all_obs()
except Exception as e:
    print(f"Error during fetch: {e}")

===== Starting MongoDB Fetch =====
Retrieved 23611 documents with aiResults.confAnimal > 0.75 and < 1.0.
CPU times: user 82.7 ms, sys: 27.1 ms, total: 110 ms
Wall time: 1.58 s


This includes a lot of blanks - that's ok because when we create a list of items to download, we will limit that category to only 200, randomly selected, stratified across deployments.

The deployments will need to be parsed from the publicurl

In [29]:
# Flatten to dataframe
flat_rows = []

for item in obs_json:
    media_id = item.get('mediaID')
    url = item.get('publicURL')
    
    flat_rows.append({
        'mediaID': media_id,
        'publicURL': url,
    })

# Create a flat DataFrame
df = pd.DataFrame(flat_rows)
df = df[~df['publicURL'].isna()] # Clean up 

# with pd.option_context('display.width', 0, 'display.max_colwidth', None):
display(df.head())

,mediaID,publicURL
0,9a6f3bbe7d62565c2ce5b632c0dfad55,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0160.JPG
1,c8bc6b3e4f8859a06ae30bf269682a27,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-31_LearningPlatformBeaver/DCIM/100MEDIA/SYFW0075.JPG
2,5aaa2811d63e67432dc693b59468f65b,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-31_LearningPlatformBeaver/DCIM/100MEDIA/SYFW0077.JPG
3,e5258a067d41e5cb16fcd368501d99a5,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-31_LearningPlatformBeaver/DCIM/100MEDIA/SYFW0079.JPG
4,924d4e2435eb699c91716c1596f8c636,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-31_LearningPlatformBeaver/DCIM/100MEDIA/SYFW0081.JPG


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 23610 entries, 0 to 23609
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   mediaID    23610 non-null  object
 1   publicURL  23610 non-null  object
dtypes: object(2)
memory usage: 553.4+ KB


We have all the pieces we need to download images.  

Cropping and resizing before running the trained mode (following how the model was trained) is next.  

In [31]:
# Load a dict for the classes
species_dict = {
    "Colorful_Songbirds_Shorbirds_Thrushlike": "Colorful Songbirds & Thrushes",
    "Gulls": "Gulls",
    "Iridescent_Blackbirds": "Iridescent Blackbirds",
    "Castor_canadensis": "Beaver",
    "Procyon_lotor": "Raccoon",
    "Sylvilagus_floridanus": "Eastern Cottontail",
    "Canis_familiaris": "Domestic Dog",
    "Ardea_herodias": "Great Blue Heron",
    "blank": "Blank",
    "Anas_platyrhynchos": "Mallard",
    "Rattus_norvegicus": "Norway Rat",
    "Nycticorax_nycticorax": "Black-crowned Night Heron",
    "Small_Brown_Grey_Songbirds": "Small Brown/Grey Songbirds",
    "Branta_canadensis": "Canada Goose",
    "Tortises": "Turtles",
    "Canis_latrans": "Coyote",
    "Felis_catus": "Domestic Cat",
    "Didelphis_virginiana": "Virginia Opossum"
}

In [7]:
# Load functions for image processing
# Custom image trimming class
class CropTopBottom(Transform):
    def __init__(self, top_pct=0.1, bottom_pct=0.1):
        self.top_pct = top_pct
        self.bottom_pct = bottom_pct

    def encodes(self, img: PILImage):
        w, h = img.size
        top = int(h * self.top_pct)
        bottom = int(h * (1 - self.bottom_pct))
        return img.crop((0, top, w, bottom))

# doc(load_learner)
learn_inf = load_learner("/kaggle/input/ur-class-convnext-expanded/pytorch/v1/1/2025-06-27-convnet_tiny_xmin2max42p-v11.pkl", cpu=True)

/usr/local/lib/python3.11/dist-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


In [8]:
# Make a prediction
learn_inf.predict('/kaggle/input/ur-classes-tests/Branta_canadensis.jpg')

('Branta_canadensis',
 tensor(2),
 tensor([5.5093e-03, 3.1224e-03, 6.6350e-01, 4.7001e-03, 1.4145e-02, 1.6052e-01,
         1.8190e-04, 1.8008e-02, 2.1816e-03, 1.9991e-03, 7.6501e-04, 6.0868e-02,
         2.3804e-02, 1.0339e-02, 2.0579e-03, 2.5835e-02, 3.9817e-04, 2.0701e-03]))

In [9]:
# Process a prediction function
def top_3_preds(image_path):
    pred_class, pred_idx, pred_probs = learn_inf.predict(image_path)

    # Convert vocab class name to common name
    def common_name(name):
        return species_dict.get(name, name)

    top3 = [
        f"{common_name(learn_inf.dls.vocab[i])}: {float(pred_probs[i]):.4f}"
        for i in pred_probs.argsort(descending=True)[:3]
    ]

    results = {
        "image_path": image_path,
        "prediction": {
            "class": str(pred_class),
            "common_name": common_name(str(pred_class)),
            "probability": round(float(pred_probs[pred_idx]), 4)
        },
        "top3": top3
    }

    return results

In [10]:
result = top_3_preds('/kaggle/input/ur-classes-tests/Branta_canadensis.jpg')
display(result)

{'image_path': '/kaggle/input/ur-classes-tests/Branta_canadensis.jpg',
 'prediction': {'class': 'Branta_canadensis',
  'common_name': 'Canada Goose',
  'probability': 0.6635},
 'top3': ['Canada Goose: 0.6635',
  'Beaver: 0.1605',
  'Black-crowned Night Heron: 0.0609']}

In [33]:
%%time
# Configuration for Multithreading and Batching
num_batches = 10
max_threads = 20

# Prepare folders
output_root = Path("output")
output_root.mkdir(exist_ok=True)
images_root = Path("images")
images_root.mkdir(exist_ok=True)

# Optional - define chunks - for each run, the first n rows will be processed
df_download = df
print(f'Peparing to Download {len(df_download)} images')

# Create a tool for resizing so cropping top and bottom can happen while keeping the aspect ratio
def resize_to_height(image, target_height=256):
    og_width, og_height = image.size
    new_width = int(og_width * (target_height / og_height))
    return image.resize((new_width, target_height))

# Tool for download and processing
def process_row(row, dest_folder, session):
    url = row['publicURL']
    filename = f"{row['mediaID']}.jpg"
    dest = dest_folder / filename

    try:
        response = session.get(url, timeout=5)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content)).convert("RGB")
        image = resize_to_height(image, target_height=256)
        image.save(dest, format="JPEG", quality=75)
    except Exception as e:
        print(f"failed to process {filename}: {e}")

for batch_idx, df_chunk in enumerate(np.array_split(df_download, num_batches)):
    batch_folder = images_root / f'batch_{batch_idx}'
    batch_folder.mkdir(exist_ok=True)
    print(f'Processing batch {batch_idx + 1} / {num_batches}...')

    rows = df_chunk.to_dict(orient='records')
    start = time.time()

    with requests.Session() as session:
        with ThreadPoolExecutor(max_workers=max_threads) as executor:
            futures = [executor.submit(process_row, row, batch_folder, session) for row in rows]
            for future in as_completed(futures):
                future.result()  # you can add error catching here if needed

    print(f"Batch {batch_idx+1} took {time.time() - start:.2f} seconds.")
        
print(f'{len(df_download)} Images Downloaded and Resized')

Peparing to Download 50 images
Processing batch 1 / 10...


/usr/local/lib/python3.11/dist-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Batch 1 took 2.62 seconds.
Processing batch 2 / 10...
Batch 2 took 2.56 seconds.
Processing batch 3 / 10...
Batch 3 took 2.60 seconds.
Processing batch 4 / 10...
Batch 4 took 2.53 seconds.
Processing batch 5 / 10...
Batch 5 took 2.82 seconds.
Processing batch 6 / 10...
Batch 6 took 2.84 seconds.
Processing batch 7 / 10...
Batch 7 took 8.90 seconds.
Processing batch 8 / 10...
Batch 8 took 2.97 seconds.
Processing batch 9 / 10...
Batch 9 took 2.64 seconds.
Processing batch 10 / 10...
Batch 10 took 2.57 seconds.
50 Images Downloaded and Resized
CPU times: user 33.1 s, sys: 9.55 s, total: 42.7 s
Wall time: 33.1 s


In [34]:
all_results = []
images_root = Path("images")

# Loop through each batch folder
for batch_folder in sorted(images_root.glob("batch_*")):
    print(f"Processing predictions in {batch_folder}...")
    
    # Loop through each image in the batch folder
    for image_file in tqdm(sorted(batch_folder.glob("*.jpg"))):
        try:
            result = top_3_preds(str(image_file))
            media_id = Path(image_file).stem
            result['mediaID'] = media_id
            all_results.append(result)
        except Exception as e:
            print(f"Failed to predict {image_file.name}: {e}")

Processing predictions in images/batch_0...


  0%|          | 0/5 [00:00<?, ?it/s]

Processing predictions in images/batch_1...


  0%|          | 0/5 [00:00<?, ?it/s]

Processing predictions in images/batch_2...


...

  0%|          | 0/5 [00:00<?, ?it/s]

Processing predictions in images/batch_8...


  0%|          | 0/5 [00:00<?, ?it/s]

Processing predictions in images/batch_9...


  0%|          | 0/5 [00:00<?, ?it/s]

In [35]:
output_file = Path("output") / "all_predictions.json"
with open(output_file, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"Saved {len(all_results)} predictions to {output_file}")

Saved 50 predictions to output/all_predictions.json


In [37]:
# Convert results to DataFrame
pred_df = pd.DataFrame([{
    'mediaID': r['mediaID'],
    'predicted_class': r['prediction']['class'],
    'common_name': r['prediction']['common_name'],
    'probability': r['prediction']['probability'],
    'top3': ", ".join(r['top3'])
} for r in all_results])

# Make sure mediaID is string in both
df['mediaID'] = df['mediaID'].astype(str)
pred_df['mediaID'] = pred_df['mediaID'].astype(str)

# Merge with original df to restore publicURL
merged_df = pred_df.merge(df[['mediaID', 'publicURL']], on='mediaID', how='left')

# Optional: Save
merged_df.to_csv(output_root / "all_predictions_with_url.csv", index=False)
print(f"CSV saved to {output_root / 'all_predictions.csv'}")

CSV saved to output/all_predictions.csv


In [40]:
display(merged_df.head())
display(merged_df['predicted_class'].value_counts())
print(f'Rows: {len(merged_df)}')

,mediaID,predicted_class,common_name,probability,top3,publicURL
0,5aaa2811d63e67432dc693b59468f65b,Canis_familiaris,Domestic Dog,0.9720,"Domestic Dog: 0.9720, Domestic Cat: 0.0198, Iridescent Blackbirds: 0.0034",https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-31_LearningPlatformBeaver/DCIM/100MEDIA/SYFW0077.JPG
1,924d4e2435eb699c91716c1596f8c636,Canis_familiaris,Domestic Dog,0.9971,"Domestic Dog: 0.9971, Mallard: 0.0015, Iridescent Blackbirds: 0.0004",https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-31_LearningPlatformBeaver/DCIM/100MEDIA/SYFW0081.JPG
2,9a6f3bbe7d62565c2ce5b632c0dfad55,Canis_familiaris,Domestic Dog,0.6394,"Domestic Dog: 0.6394, Blank: 0.1210, Mallard: 0.1040",https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0160.JPG
3,c8bc6b3e4f8859a06ae30bf269682a27,Canis_familiaris,Domestic Dog,0.9945,"Domestic Dog: 0.9945, Iridescent Blackbirds: 0.0041, Domestic Cat: 0.0005",https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-31_LearningPlatformBeaver/DCIM/100MEDIA/SYFW0075.JPG
4,e5258a067d41e5cb16fcd368501d99a5,Canis_familiaris,Domestic Dog,0.9070,"Domestic Dog: 0.9070, Canada Goose: 0.0404, Mallard: 0.0263",https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-31_LearningPlatformBeaver/DCIM/100MEDIA/SYFW0079.JPG


predicted_class
Canis_familiaris      26
Branta_canadensis     10
Anas_platyrhynchos     6
Castor_canadensis      5
Ardea_herodias         3
Name: count, dtype: int64

Rows: 50


In [42]:
display(merged_df[merged_df['predicted_class']=="Ardea_herodias"])

,mediaID,predicted_class,common_name,probability,top3,publicURL
20,0d498c7d3e1f8fcbef7d304dd6204cbb,Ardea_herodias,Great Blue Heron,0.9593,"Great Blue Heron: 0.9593, Canada Goose: 0.0139, Small Brown/Grey Songbirds: 0.0079",https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-02-01_16-41-42/DCIM/100SYCAM/SYEW1665.JPG
21,61def8a17b3b4551e190eed4027c92bc,Ardea_herodias,Great Blue Heron,0.9546,"Great Blue Heron: 0.9546, Canada Goose: 0.0145, Black-crowned Night Heron: 0.0088",https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-02-01_16-41-42/DCIM/100SYCAM/SYEW1667.JPG
23,c8ffad0ef528944528cde8d17d53de2a,Ardea_herodias,Great Blue Heron,0.9550,"Great Blue Heron: 0.9550, Canada Goose: 0.0179, Small Brown/Grey Songbirds: 0.0117",https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-02-01_16-41-42/DCIM/100SYCAM/SYEW1666.JPG
